### Mashiur Rahman

# Project 3 — Predicting Trading Signals

## Part 1 — Create Technical Indicators

I will calculate MACD and RSI separately for each stock. I will then combine their signals. The combined Buy, Sell, or Hold signal will be the target for the machine-learning models.

### Import required librarry and load dataset

In [22]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
prices = pd.read_csv(
    "stock_prepared.csv",
    usecols=["ticker", "date", "close"],
    parse_dates=["date"]
)

prices = prices.sort_values(["ticker", "date"]).reset_index(drop=True)

print("Number of records:", len(prices))
display(prices.head())

Number of records: 20860334


,ticker,close,date
0,A,32.859444,1999-12-17
1,A,33.530045,1999-12-20
2,A,33.351215,1999-12-21
3,A,34.021816,1999-12-22
4,A,35.586552,1999-12-23


### Calculate MACD

MACD is the 12-day exponential moving average (EMA) minus the 26-day EMA. The signal line is a 9-day EMA of MACD.

In [3]:
prices["ema_12"] = prices.groupby("ticker")["close"].transform(
    lambda x: x.ewm(span=12, adjust=False).mean()
)

prices["ema_26"] = prices.groupby("ticker")["close"].transform(
    lambda x: x.ewm(span=26, adjust=False).mean()
)

prices["macd"] = prices["ema_12"] - prices["ema_26"]

prices["signal_line"] = prices.groupby("ticker")["macd"].transform(
    lambda x: x.ewm(span=9, adjust=False).mean()
)

prices = prices.drop(columns=["ema_12", "ema_26"])

display(prices.head())

,ticker,close,date,macd,signal_line
0,A,32.859444,1999-12-17,0.000000,0.000000
1,A,33.530045,1999-12-20,0.053495,0.010699
2,A,33.351215,1999-12-21,0.080532,0.024666
3,A,34.021816,1999-12-22,0.154293,0.050591
4,A,35.586552,1999-12-23,0.335146,0.107502


### Create the MACD signal

In [4]:
previous_macd = prices.groupby("ticker")["macd"].shift(1)
previous_line = prices.groupby("ticker")["signal_line"].shift(1)

buy_cross = (
    (previous_macd < previous_line) &
    (prices["macd"] > prices["signal_line"])
)

sell_cross = (
    (previous_macd > previous_line) &
    (prices["macd"] < prices["signal_line"])
)

prices["macd_signal"] = np.select(
    [buy_cross, sell_cross],
    ["Buy", "Sell"],
    default="Hold"
)

display(prices[["ticker", "date", "macd", "signal_line", "macd_signal"]].head())

,ticker,date,macd,signal_line,macd_signal
0,A,1999-12-17,0.000000,0.000000,Hold
1,A,1999-12-20,0.053495,0.010699,Hold
2,A,1999-12-21,0.080532,0.024666,Hold
3,A,1999-12-22,0.154293,0.050591,Hold
4,A,1999-12-23,0.335146,0.107502,Hold


### Calculate RSI

RSI measures the strength of recent gains and losses. I will use a 14-record exponential average. RSI below 30 means Buy; RSI above 70 means Sell.

In [5]:
change = prices.groupby("ticker")["close"].diff()

gain = change.clip(lower=0)
loss = -change.clip(upper=0)

average_gain = gain.groupby(prices["ticker"]).transform(
    lambda x: x.ewm(span=14, adjust=False, min_periods=14).mean()
)

average_loss = loss.groupby(prices["ticker"]).transform(
    lambda x: x.ewm(span=14, adjust=False, min_periods=14).mean()
)

rs = average_gain / average_loss
prices["rsi"] = 100 - (100 / (1 + rs))

prices.loc[(average_loss == 0) & (average_gain > 0), "rsi"] = 100
prices.loc[(average_loss == 0) & (average_gain == 0), "rsi"] = 50

display(prices[["ticker", "date", "close", "rsi"]].head(20))

,ticker,date,close,rsi
0,A,1999-12-17,32.859444,NaN
1,A,1999-12-20,33.530045,NaN
2,A,1999-12-21,33.351215,NaN
3,A,1999-12-22,34.021816,NaN
4,A,1999-12-23,35.586552,NaN
5,A,1999-12-27,37.777184,NaN
6,A,1999-12-28,43.991417,NaN
7,A,1999-12-29,51.502148,NaN
8,A,1999-12-30,56.688126,NaN
9,A,1999-12-31,55.302216,NaN


##### Missing RSI values at the start of each stock’s history are expected: the 14-record calculation needs enough records first.

### Create the RSI signal

In [6]:
prices["rsi_signal"] = np.select(
    [prices["rsi"] < 30, prices["rsi"] > 70],
    ["Buy", "Sell"],
    default="Hold"
)

display(prices[["ticker", "date", "rsi", "rsi_signal"]].head(20))

,ticker,date,rsi,rsi_signal
0,A,1999-12-17,NaN,Hold
1,A,1999-12-20,NaN,Hold
2,A,1999-12-21,NaN,Hold
3,A,1999-12-22,NaN,Hold
4,A,1999-12-23,NaN,Hold
5,A,1999-12-27,NaN,Hold
6,A,1999-12-28,NaN,Hold
7,A,1999-12-29,NaN,Hold
8,A,1999-12-30,NaN,Hold
9,A,1999-12-31,NaN,Hold


### Create the final trading signal

The final signal is Buy when both indicators say Buy. It is Sell when both say Sell. Every other combination is Hold.

#### Combine the signals

In [7]:
both_buy = (
    (prices["macd_signal"] == "Buy") &
    (prices["rsi_signal"] == "Buy")
)

both_sell = (
    (prices["macd_signal"] == "Sell") &
    (prices["rsi_signal"] == "Sell")
)

prices["trading_signal"] = np.select(
    [both_buy, both_sell],
    ["Buy", "Sell"],
    default="Hold"
)

display(
    prices[
        ["ticker", "date", "macd_signal", "rsi_signal", "trading_signal"]
    ].head(20)
)

,ticker,date,macd_signal,rsi_signal,trading_signal
0,A,1999-12-17,Hold,Hold,Hold
1,A,1999-12-20,Hold,Hold,Hold
2,A,1999-12-21,Hold,Hold,Hold
3,A,1999-12-22,Hold,Hold,Hold
4,A,1999-12-23,Hold,Hold,Hold
5,A,1999-12-27,Hold,Hold,Hold
6,A,1999-12-28,Hold,Hold,Hold
7,A,1999-12-29,Hold,Hold,Hold
8,A,1999-12-30,Hold,Hold,Hold
9,A,1999-12-31,Hold,Hold,Hold


### Trading signal

In [8]:
counts = prices["trading_signal"].value_counts()
percentages = prices["trading_signal"].value_counts(normalize=True) * 100

summary = pd.DataFrame({
    "Count": counts,
    "Percentage": percentages.round(2)
}).reindex(["Buy", "Sell", "Hold"], fill_value=0)

display(summary)
print("Most frequent signal:", counts.idxmax())

,Count,Percentage
trading_signal,,
Buy,6548,0.03
Sell,6901,0.03
Hold,20846885,99.94


Most frequent signal: Hold


In [9]:
print("Records used:", len(prices))

Records used: 20860334


In [10]:
print("CSV size (GB):", round(os.path.getsize("stock_prepared.csv") / 1_000_000_000, 2))

with open("stock_prepared.csv", encoding="utf-8") as file:
    csv_rows = sum(1 for line in file) - 1

print("Rows in CSV:", csv_rows)
print("Rows in prices:", len(prices))

CSV size (GB): 5.93
Rows in CSV: 20860334
Rows in prices: 20860334


### Part 1 — Answers

1. **How many Buy, Sell, and Hold signals were generated?**  
   Buy: 6,548. Sell: 6,901. Hold: 20,846,885.

2. **Which signal occurs most frequently?**  
   Hold occurs most frequently, with 20,846,885 records (99.94%).

3. **Why are Buy and Sell much less common than Hold?**  
   A Buy signal requires both MACD and RSI to say Buy on the same record. A Sell signal also requires both indicators to agree. Every other combination becomes Hold, so Buy and Sell are rare.

4. **What does this tell me about the target variable?**  
  The target variable is highly imbalanced: almost all records are Hold, while very few are Buy or Sell. In the modelling steps, I will need to check how well the model identifies each signal, rather than relying only on overall accuracy.

## Part 2 — Check for Class Imbalance

In [11]:
counts = prices["trading_signal"].value_counts()
percentages = prices["trading_signal"].value_counts(normalize=True) * 100

class_summary = pd.DataFrame({
    "Count": counts,
    "Percentage": percentages.round(2)
}).reindex(["Buy", "Hold", "Sell"])

display(class_summary)

,Count,Percentage
trading_signal,,
Buy,6548,0.03
Hold,20846885,99.94
Sell,6901,0.03


### Part 2 — Answer

The target variable is highly imbalanced. Out of 20,860,334 records, 20,846,885 are Hold (99.94%). Only 6,548 are Buy (0.03%) and 6,901 are Sell (0.03%).

This could cause a model to predict Hold most of the time and still get a high accuracy score. That would not show whether it can identify Buy and Sell correctly.

I will keep the test data representative of the original data. After splitting the data, I will address the imbalance using the training data only. I will check precision, recall, and F1-score for each class and use a confusion matrix, instead of relying on accuracy alone.

## Part 3 — Prepare the Data for Machine Learning

In [13]:
columns = [
    "ticker", "date", "close", "high", "low",
    "daily_return", "close_lag_1", "ma_5",
    "ma_20", "volatility_20"
]

prepared = pd.read_csv(
    "stock_prepared.csv",
    usecols=columns,
    parse_dates=["date"]
)

prepared["daily_range_pct"] = (
    (prepared["high"] - prepared["low"]) / prepared["close"] * 100
)

print("Rows loaded:", len(prepared))

Rows loaded: 20860334


In [14]:
model_data = prepared.merge(
    prices[["ticker", "date", "trading_signal"]],
    on=["ticker", "date"]
)

features = [
    "daily_return", "close_lag_1", "ma_5",
    "ma_20", "volatility_20", "daily_range_pct"
]

model_data = model_data.dropna(subset=features)
model_data["target"] = model_data["trading_signal"].map({
    "Buy": 0, "Hold": 1, "Sell": 2
})

print("Rows ready for modelling:", len(model_data))
display(model_data[["ticker", "date"] + features + ["trading_signal", "target"]].head())

Rows ready for modelling: 20860334


,ticker,date,daily_return,close_lag_1,ma_5,ma_20,volatility_20,daily_range_pct,trading_signal,target
0,A,1999-12-17,-0.027778,33.798283,31.956366,31.209764,0.051155,3.673467,Hold,1
1,A,1999-12-20,0.020408,32.859444,32.153077,31.442239,0.047086,1.733336,Hold,1
2,A,1999-12-21,-0.005333,33.530045,32.671674,31.536123,0.043080,1.474536,Hold,1
3,A,1999-12-22,0.020107,33.351215,33.512160,31.806599,0.036956,2.628121,Hold,1
4,A,1999-12-23,0.045992,34.021816,33.869814,32.117310,0.037681,5.150758,Hold,1


### Part 3 — Answer

I used six features: daily_return, close_lag_1, ma_5, ma_20, volatility_20, and daily_range_pct. I matched them with the trading signals using ticker and date, then removed rows with missing feature values.

I changed the target labels to numbers: Buy = 0, Hold = 1, and Sell = 2. I did not use MACD or RSI as model features because they were used to create the trading signals.

## Part 4 — Split the Data

#### Split by date:

In [15]:
cutoff = model_data["date"].quantile(0.80)

train = model_data[model_data["date"] <= cutoff]
test = model_data[model_data["date"] > cutoff]

X_train = train[features]
X_test = test[features]

y_train = train["target"]
y_test = test["target"]

print("Last training date:", train["date"].max().date())
print("First test date:", test["date"].min().date())
print("Training rows:", len(train))
print("Test rows:", len(test))

print("\nTraining signals:")
print(train["trading_signal"].value_counts())

print("\nTest signals:")
print(test["trading_signal"].value_counts())

Last training date: 2015-05-08
First test date: 2015-05-11
Training rows: 16690595
Test rows: 4169739

Training signals:
trading_signal
Hold    16680277
Sell        5356
Buy         4962
Name: count, dtype: int64

Test signals:
trading_signal
Hold    4166608
Buy        1586
Sell       1545
Name: count, dtype: int64


#### Standardize the features:

In [17]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

Training shape: (16690595, 6)
Test shape: (4169739, 6)


### Part 4 — Answer

I split the data by date. Earlier records are in the training set, and later records are in the test set. I did not shuffle the data because stock prices are time-based.

I fitted the StandardScaler on the training features only and used it to transform both sets. The test set was left in its original class distribution so it represents the data the model will face.

## Part 5 — Build Machine Learning Models

#### Select training records:

In [18]:
rare_signals = train[train["target"] != 1]
hold_signals = train[train["target"] == 1].sample(
    n=20000,
    random_state=42
)

train_sample = pd.concat([rare_signals, hold_signals])

X_sample = scaler.transform(train_sample[features])
y_sample = train_sample["target"]

print("Training sample size:", len(train_sample))
print(train_sample["trading_signal"].value_counts())

Training sample size: 30318
trading_signal
Hold    20000
Sell     5356
Buy      4962
Name: count, dtype: int64


#### Train the three models:

In [20]:
logistic_model = LogisticRegression(max_iter=1000, class_weight="balanced")
forest_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=12,
    random_state=42,
    n_jobs=2
)
svm_model = LinearSVC(class_weight="balanced", max_iter=2000)

logistic_model.fit(X_sample, y_sample)
print("Logistic Regression trained")

forest_model.fit(train_sample[features], y_sample)
print("Random Forest trained")

svm_model.fit(X_sample, y_sample)
print("Linear SVM trained")

Logistic Regression trained
Random Forest trained
Linear SVM trained


### Part 5 — Explanation

I trained Logistic Regression, Random Forest, and a linear SVM.

I used 30,318 training records: 4,962 Buy, 5,356 Sell, and 20,000 Hold. I kept all Buy and Sell records from the training set and randomly selected 20,000 Hold records using random_state=42. I used a sample because the full dataset is very large for my computer.

I used scaled features for Logistic Regression and SVM. Random Forest used the original features. I did not change the test set.

## Part 6 — Evaluate the Models

#### Select test records:

In [21]:
test_sample = test.sample(n=min(500000, len(test)), random_state=42)

X_test_sample = test_sample[features]
X_test_sample_scaled = scaler.transform(X_test_sample)
y_test_sample = test_sample["target"]

print("Test records evaluated:", len(test_sample))
print(test_sample["trading_signal"].value_counts())

Test records evaluated: 500000
trading_signal
Hold    499632
Buy        191
Sell       177
Name: count, dtype: int64


#### Evaluate all three models:

In [23]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = [
    ("Logistic Regression", logistic_model, X_test_sample_scaled),
    ("Random Forest", forest_model, X_test_sample),
    ("Linear SVM", svm_model, X_test_sample_scaled)
]

results = []

for name, model, X in models:
    predictions = model.predict(X)

    report = classification_report(
        y_test_sample,
        predictions,
        labels=[0, 1, 2],
        target_names=["Buy", "Hold", "Sell"],
        zero_division=0,
        output_dict=True
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_sample, predictions),
        "Precision (macro)": report["macro avg"]["precision"],
        "Recall (macro)": report["macro avg"]["recall"],
        "F1 (macro)": report["macro avg"]["f1-score"]
    })

    print("\n", name)
    print(classification_report(
        y_test_sample,
        predictions,
        labels=[0, 1, 2],
        target_names=["Buy", "Hold", "Sell"],
        zero_division=0
    ))

    matrix = confusion_matrix(y_test_sample, predictions, labels=[0, 1, 2])
    display(pd.DataFrame(
        matrix,
        index=["Actual Buy", "Actual Hold", "Actual Sell"],
        columns=["Predicted Buy", "Predicted Hold", "Predicted Sell"]
    ))

comparison = pd.DataFrame(results).set_index("Model").round(4)
display(comparison)


 Logistic Regression
              precision    recall  f1-score   support

         Buy       0.00      0.12      0.01       191
        Hold       1.00      0.70      0.82    499632
        Sell       0.00      0.11      0.00       177

    accuracy                           0.69    500000
   macro avg       0.33      0.31      0.28    500000
weighted avg       1.00      0.69      0.82    500000



,Predicted Buy,Predicted Hold,Predicted Sell
Actual Buy,23,87,81
Actual Hold,6852,347377,145403
Actual Sell,0,157,20



 Random Forest
              precision    recall  f1-score   support

         Buy       0.01      0.28      0.01       191
        Hold       1.00      0.96      0.98    499632
        Sell       0.00      0.20      0.00       177

    accuracy                           0.96    500000
   macro avg       0.34      0.48      0.33    500000
weighted avg       1.00      0.96      0.98    500000



,Predicted Buy,Predicted Hold,Predicted Sell
Actual Buy,54,135,2
Actual Hold,7084,477573,14975
Actual Sell,4,137,36



 Linear SVM
              precision    recall  f1-score   support

         Buy       0.01      0.01      0.01       191
        Hold       1.00      1.00      1.00    499632
        Sell       0.00      0.00      0.00       177

    accuracy                           1.00    500000
   macro avg       0.34      0.34      0.34    500000
weighted avg       1.00      1.00      1.00    500000



,Predicted Buy,Predicted Hold,Predicted Sell
Actual Buy,2,189,0
Actual Hold,191,499388,53
Actual Sell,0,177,0


,Accuracy,Precision (macro),Recall (macro),F1 (macro)
Model,,,,
Logistic Regression,0.6948,0.3343,0.3096,0.2756
Random Forest,0.9553,0.3365,0.4807,0.3322
Linear SVM,0.9988,0.3365,0.3367,0.3366


## Part 7 — Improve Random Forest

#### Train the changed model:

In [25]:
improved_forest = RandomForestClassifier(
    n_estimators=50,
    max_depth=16,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=2
)

improved_forest.fit(train_sample[features], y_sample)
print("Improved Random Forest trained")

Improved Random Forest trained


#### Compare before and after:

In [26]:
original_pred = forest_model.predict(X_test_sample)
improved_pred = improved_forest.predict(X_test_sample)

comparison_rows = []

for name, predictions in [
    ("Original", original_pred),
    ("Changed", improved_pred)
]:
    report = classification_report(
        y_test_sample,
        predictions,
        labels=[0, 1, 2],
        target_names=["Buy", "Hold", "Sell"],
        zero_division=0,
        output_dict=True
    )

    for signal in ["Buy", "Hold", "Sell"]:
        comparison_rows.append({
            "Model": name,
            "Signal": signal,
            "Precision": report[signal]["precision"],
            "Recall": report[signal]["recall"],
            "F1-score": report[signal]["f1-score"]
        })

    print(name, "accuracy:", round(accuracy_score(y_test_sample, predictions), 4))

part7_results = pd.DataFrame(comparison_rows).round(4)
display(part7_results)

Original accuracy: 0.9553
Changed accuracy: 0.7191


,Model,Signal,Precision,Recall,F1-score
0,Original,Buy,0.0076,0.2827,0.0147
1,Original,Hold,0.9994,0.9558,0.9772
2,Original,Sell,0.0024,0.2034,0.0047
3,Changed,Buy,0.0037,0.5288,0.0073
4,Changed,Hold,0.9996,0.7192,0.8365
5,Changed,Sell,0.0010,0.6554,0.0021


#### Confusion matrix for the changed model:

In [27]:
matrix = confusion_matrix(y_test_sample, improved_pred, labels=[0, 1, 2])

display(pd.DataFrame(
    matrix,
    index=["Actual Buy", "Actual Hold", "Actual Sell"],
    columns=["Predicted Buy", "Predicted Hold", "Predicted Sell"]
))

,Predicted Buy,Predicted Hold,Predicted Sell
Actual Buy,101,71,19
Actual Hold,27551,359329,112752
Actual Sell,6,55,116


### Part 7 — What I Changed

I changed two Random Forest settings. I increased max_depth from 12 to 16 and added class_weight="balanced_subsample" to give the less common Buy and Sell records more weight during training.

I trained the changed model on the same training sample and evaluated it on the same test sample as the original model. I compared accuracy and the precision, recall, and F1-score for Buy, Hold, and Sell.

## Part 8 — Reflect on the Results
1. Which model performed differently across Buy, Hold, and Sell?
All three performed much better on Hold than on Buy or Sell. Random Forest found more Buy and Sell records than the other original models: Buy recall was 28.27% and Sell recall was 20.34%. Its Hold recall was 95.58%. Linear SVM almost always predicted Hold; it found just 2 of 191 Buy records and 0 of 177 Sell records.

2.  Did high accuracy correspond to good performance on Buy and Sell?
No. Linear SVM reached 99.88% accuracy but identified only 2 Buy records and no Sell records in the 500,000-row test sample. The sample contained 499,632 Hold records, so accuracy alone hid this poor result on rare signals.


3. How did class imbalance affect the results?
Hold dominated the original data: 20,846,885 of 20,860,334 records (99.94%). The model could score well by predicting Hold most of the time. Buy and Sell precision, recall, F1, and the confusion matrices were needed to understand its performance.


4. What happened when you addressed the imbalance?
For training, I kept all Buy and Sell records and sampled 20,000 Hold records. I then added class weights in the changed Random Forest. Compared with the original Random Forest, Buy recall rose from 28.27% to 52.88% and Sell recall from 20.34% to 65.54%. However, Buy and Sell precision and F1 fell, and overall accuracy dropped from 95.53% to 71.91%. The changed model made too many false Buy and Sell predictions.


5. Which features might be useful for predicting the signals?
In the original Random Forest, daily_return had the highest feature importance (0.3069), followed by volatility_20 (0.1979) and ma_20 (0.1466). This suggests they were useful to this model. Feature importance alone does not prove that any feature causes the signals or predicts profits.


6. What are the limitations of using MACD/RSI-based rules to create the target?
Buy and Sell labels depend on the chosen MACD crossover and RSI thresholds. These rules made the target very imbalanced. A rule-based label describes what the indicators said; it does not prove that buying or selling would have been profitable. The models are learning to predict those labels rather than actual future returns.


7. What additional features could be explored in a future project?
I could test trading-volume changes, longer-term price trends, market-index returns, and sector information. I would calculate each feature using only information available at the time of prediction and test whether it improves Buy and Sell precision and recall.


8. Would these model predictions guarantee profitable trading? Why or why not?
No. The predictions had many false Buy and Sell signals, and this project did not test actual trade returns. Fees, slippage, changing markets, and risk can reduce or eliminate profits. A time-based trading backtest would be needed to study possible performance; even a good backtest would not guarantee future profits.